# ID10M — System G BIO Tagger Training

Trains System G (mBERT BIO tagger) on ID10M training data.  
Evaluates on ID10M test — gives BIO F1 directly comparable to paper Table 5.

**Pipeline:**
1. Clone repo + mount Drive
2. Clone ID10M dataset
3. Convert ID10M TSV → JSONL
4. Train BiO_Task_mBERT_train.py on ID10M train split
5. Eval on ID10M test split
6. Save model + results to Drive

In [ ]:
# ── CELL 1: Mount Drive + clone repo ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys, os, shutil

REPO     = 'https://github.com/JustLetMeBeHello/Idiomator_Research.git'
REPO_DIR = '/content/Research_And_Training'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', REPO, REPO_DIR], check=True)

os.makedirs('/content/drive/MyDrive/Idiomator_Research/models',  exist_ok=True)
os.makedirs('/content/drive/MyDrive/Idiomator_Research/results', exist_ok=True)

# Remove real committed dirs FIRST, then symlink (ln -sfn won't replace a real dir)
for d in ['models', 'results']:
    p = os.path.join(REPO_DIR, d)
    if os.path.islink(p):
        os.unlink(p)
    elif os.path.isdir(p):
        shutil.rmtree(p)
    os.symlink(f'/content/drive/MyDrive/Idiomator_Research/{d}', p)

# HARD GATE — both must be True or STOP
ok = True
for d in ['models', 'results']:
    p = os.path.join(REPO_DIR, d)
    is_link = os.path.islink(p)
    ok &= is_link
    print(d, 'islink:', is_link, '->', os.readlink(p) if is_link else '(REAL DIR - BAD)')
assert ok, 'Symlinks NOT set up — STOP, do not run training cells.'
print('\nGate passed. Safe to proceed.')

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'Requirements.txt'], check=True)
!nvidia-smi

In [ ]:
# ── CELL 2: Clone ID10M dataset ───────────────────────────────────────────────
!git clone https://github.com/Babelscape/ID10M /tmp/id10m

# Verify splits exist
from pathlib import Path
for lang in ['english', 'spanish']:
    for split in ['train', 'test']:
        p = Path(f'/tmp/id10m/resources/bio_format/{lang}/{split}_{lang}.tsv')
        print(f'  {p.name}: {"✓" if p.exists() else "MISSING"}')

In [ ]:
# ── CELL 3: Convert ID10M TSV → JSONL ────────────────────────────────────────
# Output goes to Drive for persistence
JSONL_DIR = Path('/content/drive/MyDrive/IdiomatorRigor/id10m_jsonl')
JSONL_DIR.mkdir(parents=True, exist_ok=True)

!python3 -u Additional_Rigor_Experiments/id10m_to_jsonl.py \
    --data_dir  /tmp/id10m \
    --output_dir {JSONL_DIR} \
    --langs EN ES \
    --splits train dev test

# Verify
import json
for split in ['train', 'dev', 'test']:
    p = JSONL_DIR / f'{split}.jsonl'
    if p.exists():
        n = sum(1 for _ in p.open())
        sample = json.loads(p.open().readline())
        print(f'  {split}.jsonl: {n} rows | sample span={sample.get("span_start")}-{sample.get("span_end")} idiom={sample.get("idiom")!r}')

In [ ]:
# ── CELL 4: Train System G on ID10M ──────────────────────────────────────────
# Uses same hyperparams as MultiIdiom System G (tuned via Optuna)
# ~1-2h on T4 depending on dataset size

MODEL_OUT = Path('/content/drive/MyDrive/IdiomatorModels/system_g_id10m')
MODEL_OUT.mkdir(parents=True, exist_ok=True)

# Verify output is on Drive (persistence gate)
probe = MODEL_OUT / '.write_test'
probe.write_text('ok')
assert probe.read_text() == 'ok', 'OUTPUT NOT DURABLE — check symlinks'
probe.unlink()
print('✓ Output dir durable')

!python3 -u Ablations/BiO_Task_mBERT_train.py \
    --data_dir      {JSONL_DIR} \
    --output_dir    {MODEL_OUT} \
    --langs         English Spanish \
    --test_langs    English Spanish \
    --model_name    bert-base-multilingual-cased \
    --epochs        6 \
    --batch_size    32 \
    --lr            3.27e-5 \
    --dropout       0.2394 \
    --o_weight      0.104 \
    --device        cuda

In [ ]:
# ── CELL 5: Results ───────────────────────────────────────────────────────────
import json
from pathlib import Path

MODEL_OUT = Path('/content/drive/MyDrive/IdiomatorModels/system_g_id10m')
metrics_path = MODEL_OUT / 'metrics.json'

assert metrics_path.exists(), f'metrics.json not found — training may have failed'
m = json.loads(metrics_path.read_text())
print(json.dumps(m, indent=2))

# Key numbers for paper comparison
print('\n=== ID10M Comparison ===')
print('ID10M paper (supervised, Table 5):')
print('  EN BIO macro F1: ~0.80  (from paper — verify exact value)')
print('  ES BIO macro F1: ~0.77  (from paper — verify exact value)')
print('\nSystem G trained on ID10M (our supervised):')
for lang, v in m.get('per_lang', {}).items():
    print(f'  {lang}: span_overlap_f1={v.get("overlap_f1", "N/A")}  exact={v.get("exact_match", "N/A")}')

print('\nSystem E zero-shot BIO F1 (from ID10M_BIO_Diagnostic):')
print('  EN: 0.6108')
print('  ES: 0.6266')